# Shared-checkpoint CIFAR-100 comparison

The exact model, optimizer, data split, loader generator, and RNG state at
`theta_300` are hashed and forked into four 60-epoch arms. Every arm rebases
the LR stored at theta_300 onto the same 60-epoch cosine segment ending at 0:

- adaptive-site `ours_e_driven_o` (GPU 0) and relaxed Bypass (GPU 1),
  concurrently;
- `vanilla_continue` (GPU 0) and `o_projection_only` (GPU 1) in wave 2.

The total budget is 360 epochs for every arm. The official CIFAR-100 test set is
never constructed. Every process saves a resumable checkpoint each epoch.
Bypass spends 40 epochs in opt1 and at most 20 in opt2. It never force-projects
a nonzero D; an uncontracted run is retained with `bypass_completed=false`.


In [ ]:
import hashlib, json, os, shutil, subprocess, sys, threading
from pathlib import Path
from kaggle_secrets import UserSecretsClient

MAIN_URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
REPO = Path("/kaggle/working/counterfactual-projection")
REFERENCE_URL = "https://github.com/duyh80456-code/One-Shot-TAS-CCIL.git"
REFERENCE = Path("/kaggle/working/One-Shot-TAS-CCIL")
GROMO_URL = "https://github.com/growingnet/gromo.git"
GROMO_COMMIT = "8d19107b61a9459a9021065a329b699adcb0f25b"
GROMO = Path("/kaggle/working/gromo")
OUTPUT = Path("/kaggle/working/counterfactual_shared_theta300_360ep_all8_v8")

def private_clone(url, destination, branch):
    token = UserSecretsClient().get_secret("github_token").strip()
    if not token: raise RuntimeError("Kaggle Secret github_token is unavailable")
    askpass = Path("/kaggle/working/.counterfactual_git_askpass.py")
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass),
               GIT_TERMINAL_PROMPT="0")
    try:
        subprocess.run(["git", "clone", "--branch", branch, "--single-branch",
                        url, str(destination)], env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)
        env.pop("GITHUB_TOKEN_RUNTIME", None)
        token = None

for checkout in (REPO, REFERENCE, GROMO):
    if checkout.exists(): shutil.rmtree(checkout)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, "main")
private_clone(REFERENCE_URL, REFERENCE, "ccil-residual-capacity")
subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                GROMO_URL, str(GROMO)], check=True)
subprocess.run(["git", "-C", str(GROMO), "fetch", "--depth", "1",
                "origin", GROMO_COMMIT], check=True)
subprocess.run(["git", "-C", str(GROMO), "checkout", "--detach",
                GROMO_COMMIT], check=True)
if subprocess.check_output(["git", "-C", str(GROMO), "rev-parse", "HEAD"],
                           text=True).strip() != GROMO_COMMIT:
    raise RuntimeError("Gromo revision mismatch")
print("main:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())
print("reference:", subprocess.check_output(
    ["git", "-C", str(REFERENCE), "rev-parse", "HEAD"], text=True).strip())
print("gromo:", GROMO_COMMIT)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(GROMO)], check=True)
GROMO_SRC = GROMO / "src"
RUNTIME_PYTHONPATH = os.pathsep.join(filter(None, (
    str(REPO), str(GROMO_SRC), str(REFERENCE), os.environ.get("PYTHONPATH", ""))))
test_env = os.environ.copy()
test_env.update(PYTHONPATH=RUNTIME_PYTHONPATH, REQUIRE_GROMO_INTEGRATION="1")
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO,
               env=test_env, check=True)

import torch
if torch.cuda.device_count() != 2:
    raise RuntimeError(f"Select Kaggle T4 x2; found {torch.cuda.device_count()} GPU(s)")
print(subprocess.check_output(
    ["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
    text=True))
cifar_dirs = sorted({p.parent.resolve()
                     for p in Path("/kaggle/input").rglob("cifar-100-python")})
if not cifar_dirs:
    raise FileNotFoundError(
        "Attach a Kaggle CIFAR-100 dataset containing cifar-100-python")
DATA_ROOT = cifar_dirs[0]
print("CIFAR-100 root:", DATA_ROOT)


In [ ]:
SEED = 1
FORK_EPOCH = 300
TOTAL_EPOCHS = 360
POST_FORK_EPOCHS = 60
BATCH_SIZE = 64
VALIDATION_SAMPLES = 5000
TUNING_SAMPLES = 128
LR = 0.1
WEIGHT_DECAY = 5e-4
SHARED_CHECKPOINT = OUTPUT / "warmup" / "shared_seed1_epoch300.pt"

# A completed theta_300 checkpoint is mandatory. Never silently fall back to
# theta_150 and spend the run rebuilding the shared state.
warmups = []
for manifest_path in Path("/kaggle/input").rglob("shared_seed1_epoch300.json"):
    checkpoint_path = manifest_path.with_suffix(".pt")
    if not checkpoint_path.is_file(): continue
    manifest = json.loads(manifest_path.read_text())
    if int(manifest.get("epoch", -1)) != FORK_EPOCH: continue
    warmups.append((manifest_path, checkpoint_path, manifest))
if not warmups:
    raise FileNotFoundError(
        "Missing theta_300. Attach an output containing "
        "warmup/shared_seed1_epoch300.pt and .json; theta_150 is not accepted.")
hashes = {item[2]["sha256"] for item in warmups}
if len(hashes) != 1:
    raise RuntimeError("Multiple different theta_300 checkpoints are attached")
source_manifest, source_checkpoint, source_metadata = warmups[0]

# Restore complete progress only from the WHEN-WHERE-HOW protocol. Fixed-site
# and raw-TINY checkpoints must never be resumed as the new main algorithm.
prior_roots = [manifest.parent.parent for manifest, _, _ in warmups]
restored_adaptive = False
for prior_root in prior_roots:
    arm_checkpoint = prior_root / "ours_e_driven_o" / "checkpoint_latest.pt"
    if not arm_checkpoint.is_file(): continue
    saved_arm = torch.load(arm_checkpoint, map_location="cpu")
    functional = saved_arm.get("protocol", {}).get("functional_projection", {})
    if (functional.get("site") != "auto" or
            functional.get("site_selection_mode") !=
            "all_sites_projected_utility" or
            saved_arm.get("protocol", {}).get("total_epochs") != 360 or
            saved_arm.get("protocol", {}).get("post_fork_epochs") != 60):
        continue
    for child in prior_root.iterdir():
        destination = OUTPUT / child.name
        if destination.exists(): continue
        if child.is_dir(): shutil.copytree(child, destination)
        else: shutil.copy2(child, destination)
    print("Restored prior WHEN-WHERE-HOW run from", prior_root)
    restored_adaptive = True
    break
SHARED_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
if not SHARED_CHECKPOINT.is_file():
    shutil.copy2(source_checkpoint, SHARED_CHECKPOINT)
    shutil.copy2(source_manifest, SHARED_CHECKPOINT.with_suffix(".json"))
    print("Restored theta_300 only from", source_checkpoint)

manifest = json.loads(SHARED_CHECKPOINT.with_suffix(".json").read_text())
SHARED_HASH = manifest["sha256"]
digest = hashlib.sha256()
with SHARED_CHECKPOINT.open("rb") as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        digest.update(chunk)
if digest.hexdigest() != SHARED_HASH:
    raise RuntimeError("theta_300 checkpoint SHA-256 does not match its manifest")

def base_args(output):
    return ["--reference-root", str(REFERENCE), "--data-root", str(DATA_ROOT),
        "--output", str(output), "--shared-checkpoint", str(SHARED_CHECKPOINT),
        "--seed", str(SEED), "--batch-size", str(BATCH_SIZE),
        "--validation-samples", str(VALIDATION_SAMPLES),
        "--tuning-samples", str(TUNING_SAMPLES), "--lr", str(LR),
        "--weight-decay", str(WEIGHT_DECAY)]

if manifest["epoch"] != FORK_EPOCH:
    raise RuntimeError(f"shared checkpoint is at epoch {manifest['epoch']}")
print("theta_300 SHA-256:", SHARED_HASH)


In [ ]:
def run_wave(assignments):
    running = []

    def stream_output(name, process, log):
        for line in process.stdout:
            log.write(line)
            log.flush()
            print(f"[{name}] {line}", end="", flush=True)
        process.stdout.close()

    for gpu, name, command in assignments:
        arm_dir = OUTPUT / name
        arm_dir.mkdir(parents=True, exist_ok=True)
        log = (arm_dir / "run.log").open("a")
        process_env = os.environ.copy()
        process_env.update(CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1",
                           OMP_NUM_THREADS="2", PYTHONPATH=RUNTIME_PYTHONPATH)
        process = subprocess.Popen(
            command, cwd=REPO, env=process_env, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, encoding="utf-8",
            errors="replace", bufsize=1)
        reader = threading.Thread(
            target=stream_output, args=(name, process, log), daemon=True)
        reader.start()
        running.append((name, process, reader, log))
        print(f"GPU{gpu}: started {name}, pid={process.pid}")
    failures = []
    for name, process, reader, log in running:
        return_code = process.wait()
        reader.join()
        log.close()
        print(f"{name}: exit={return_code}")
        if return_code: failures.append((name, return_code))
    if failures:
        tails = {name: (OUTPUT / name / "run.log").read_text(
            errors="replace").splitlines()[-100:] for name, _ in failures}
        raise RuntimeError(json.dumps(
            {"failures": failures, "log_tails": tails}, indent=2))

ours = [sys.executable, "-m", "experiments.run_shared_comparison",
    "--method", "ours_e_driven_o"] + base_args(OUTPUT / "ours_e_driven_o") + [
    "--shared-checkpoint-hash", SHARED_HASH, "--site", "auto",
    "--candidate-sites", "",
    "--site-selection-mode", "all_projected_utility",
    "--selection-samples", "16",
    "--selection-cg-iterations", "25",
    "--rank", "4", "--probe-epsilon", "0.05", "--cg-iterations", "200",
    "--cg-relative-tolerance", "1e-2", "--cg-preconditioner-probes", "8"]
bypass = [sys.executable, "-m", "baselines.run_bypass"] + base_args(
    OUTPUT / "bypass") + ["--shared-checkpoint-hash", SHARED_HASH,
    "--opt1-epochs", "40", "--max-opt2-epochs", "20",
    "--contraction-epsilon", "0.002", "--gamma-slope", "3e-6"]

print("Wave 1/2: Ours + Bypass")
run_wave([(0, "ours_e_driven_o", ours), (1, "bypass", bypass)])


In [ ]:
vanilla = [sys.executable, "-m", "experiments.run_shared_comparison",
    "--method", "vanilla_continue"] + base_args(OUTPUT / "vanilla_continue") + [
    "--shared-checkpoint-hash", SHARED_HASH]
projection_only = [sys.executable, "-m", "experiments.run_shared_comparison",
    "--method", "o_projection_only"] + base_args(OUTPUT / "o_projection_only") + [
    "--shared-checkpoint-hash", SHARED_HASH, "--site", "stages.2.blocks.0",
    "--probe-epsilon", "0.05", "--cg-iterations", "200",
    "--cg-relative-tolerance", "1e-2", "--cg-preconditioner-probes", "8"]
print("Wave 2/2: Vanilla + O/projection-only")
run_wave([(0, "vanilla_continue", vanilla),
          (1, "o_projection_only", projection_only)])


In [ ]:
required = {"method", "shared_checkpoint_hash", "fork_epoch",
    "post_fork_epochs", "final_validation_accuracy", "best_validation_accuracy",
    "final_validation_loss", "best_validation_loss", "fork_validation_accuracy",
    "fork_validation_loss", "validation_accuracy_delta",
    "validation_loss_delta", "best_validation_accuracy_delta",
    "best_validation_loss_delta", "accuracy_comparison_eligible",
    "comparison_status", "training_seconds",
    "peak_gpu_memory", "deploy_params", "projection_overhead_seconds",
    "final_learning_rates"}
results = []
for name in ("ours_e_driven_o", "bypass", "vanilla_continue",
             "o_projection_only"):
    path = OUTPUT / name / "result.json"
    result = json.loads(path.read_text())
    missing = sorted(required - result.keys())
    if missing: raise RuntimeError(f"{name} missing result fields: {missing}")
    if result["shared_checkpoint_hash"] != SHARED_HASH:
        raise RuntimeError(f"{name} did not fork from theta_300")
    if result["fork_epoch"] != FORK_EPOCH or result["post_fork_epochs"] != POST_FORK_EPOCHS:
        raise RuntimeError(f"{name} did not complete the 300+60 protocol")
    if any(abs(float(lr)) > 1e-12 for lr in result["final_learning_rates"]):
        raise RuntimeError(f"{name} scheduler did not reach zero at epoch 360")
    if name == "bypass" and result.get("bypass_completed") is not True:
        print("WARNING: Bypass exhausted the matched 60-epoch budget before "
              "contraction. Accuracy is retained diagnostically but is not "
              "a completed Bypass comparator.")
    if not (OUTPUT / name / "checkpoint_latest.pt").is_file():
        raise RuntimeError(f"{name} has no resumable checkpoint")
    results.append(result)

summary = {"dataset": "CIFAR-100", "architecture": "CIFAR-ResNet18",
    "input_size": 32, "seed": SEED, "fork_epoch": FORK_EPOCH,
    "post_fork_epochs": POST_FORK_EPOCHS, "total_epochs": TOTAL_EPOCHS,
    "shared_checkpoint_hash": SHARED_HASH, "official_test_used": False,
    "completed_accuracy_comparators": [
        row["method"] for row in results
        if row["accuracy_comparison_eligible"]],
    "incomplete_comparators": [
        {"method": row["method"], "status": row["comparison_status"]}
        for row in results if not row["accuracy_comparison_eligible"]],
    "results": [{key: row.get(key) for key in sorted(required | {
        "correction_application_rate", "actual_cosine_alignment",
        "actual_relative_residual", "opt1_epochs", "opt2_epochs",
        "best_validation_loss", "fork_validation_accuracy",
        "fork_validation_loss", "validation_accuracy_delta",
        "validation_loss_delta", "best_validation_accuracy_delta",
        "best_validation_loss_delta", "accuracy_comparison_eligible",
        "comparison_status",
        "projection_overhead_seconds", "final_learning_rates",
        "train3_epochs", "contraction_norm", "projection_loss_jump",
        "contraction_criterion_met", "bypass_completed",
        "opt2_soft_cap_exceeded", "method_label", "control_type",
        "functional_target", "uses_structural_E", "site_selection_mode",
        "site_selection_history", "when_gate_pass_rate",
        "full_projection_attempt_rate"})}
        for row in results]}
(OUTPUT / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True))
print(json.dumps(summary, indent=2, sort_keys=True))
archive = shutil.make_archive(str(OUTPUT), "gztar", root_dir=OUTPUT)
print("Archive:", archive)
